In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from agent.prompt.sys import SYSTEM_PROMPT
from firecrawl import Firecrawl
from langchain_tavily import TavilySearch , TavilyExtract 
from db.vector import retriever

c:\Users\BHUMIT SINGH\Documents\AGENTIC AI\Campus Bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")


In [5]:
# from langchain_openrouter import ChatOpenRouter

# llm = ChatOpenRouter(
#     model="openai/gpt-oss-120b:free",
#     temperature=0.8,
# )


In [6]:
app = Firecrawl(api_key="fc-3d998f79373d476095948e8b1f919618")
tavily = TavilySearch()
tavily_extract=TavilyExtract(extract_depth="basic",format="markdown")


In [7]:
class GlobalState(TypedDict):
    query: str
    messages: Annotated[list[BaseMessage], add_messages]


Academic schedule
Examination updates
Events
Administrative procedures
Institutional services


In [8]:
@tool
def get_attendence(computer_code: str, password: str):
    """Tools to get the attence of a student student"""
    result = app.scrape(
        "https://cms2.ipsacademy.net/Login/sign_in", formats=["markdown"]
    )
    scrape_id = result.metadata.scrape_id
    app.interact(
        scrape_id,
        prompt=f"login using computer code - {computer_code} , pass- {password}",
    )
    response = app.interact(scrape_id, prompt="return the attendence")
    return response.output

In [9]:
@tool
def get_syllabus(
    course: Literal[
        "CSE-AIML",
        "CSE",
        "CS-IT",
        "CSE-DS",
    ],
):
    """Get syllabus link from IPS Academy. Searches official site only."""
    if course == "CSE":
        result = tavily_extract.invoke(
            {
                "urls": [
                    "https://ies.ipsacademy.org/departments/computer-science-engg/scheme-syllabus/"
                ]
            }
        )
        return result
    if(course=="CSE-AIML"):
        result = tavily_extract.invoke(
            {
                "urls": [
                    "https://ies.ipsacademy.org/departments/computer-science-engineering-artificial-intelligence-machine-learning/scheme-syllabus/"
                ]
            }
        )
        return result
    if(course=="CSE-DS"):
        result = tavily_extract.invoke(
            {
                "urls": [
                    "https://ies.ipsacademy.org/departments/scheme-syllabus-2/"
                ]
            }
        )
        return result
    if(course=="CS-IT"):
        result = tavily_extract.invoke(
            {
                "urls": [
                    "https://ies.ipsacademy.org/scheme-syllabus/"
                ]
            }
        )
        return result

In [10]:
@tool
def academic_calander():
    """Get the examination schedules and the academic calander"""
    print("Calender")
    result = tavily_extract.invoke(
        {"urls": ["https://ies.ipsacademy.org/academics/academic-calendar"]}
    )
    return result['results'][0]['raw_content']

In [40]:
@tool
def scrape_url(url: str):
    """Tool to get the content of a pdf or scrape information from a webpage"""
    print("Parsing Pdf - ",url)
    doc = app.scrape(url,formats=['markdown'])
    return doc.markdown

In [12]:
@tool
def academic_programs(course: Literal["BE/BTech", "ME/MTech"]):
    """Get the Academic programs and courses provided by institute"""
    print("getting academic programs")
    if course == "ME/MTech":
        result = tavily_extract.invoke(
            {"urls": ["https://ies.ipsacademy.org/academics/academic-programs/m-e"]}
        )
        return result['results'][0]['raw_content']
    else:
        result = tavily_extract.invoke(
            {"urls": ["https://ies.ipsacademy.org/academics/academic-programs/b-e"]}
        )
        return result['results'][0]['raw_content']


In [13]:
@tool
def upcoming_events():
    """Use this tool to get the upcoming institute events"""
    result = tavily_extract.invoke(
        {"urls": ["https://ies.ipsacademy.org/category/upcoming/"]}
    )
    return result["results"][0] 

In [14]:
@tool
def admission_procedure():
    """Tool to get the admission procedure at institute"""
    result = tavily_extract.invoke(
        {"urls": ["https://ies.ipsacademy.org/adminssion/admission-procedure/"]}
    )
    return result["results"][0]


In [49]:
@tool
def institute_brochure(query:str):
    """Brochure contains information related to and about Institute"""
    docs=retriever(query=query,namespace="campus")
    return docs

In [50]:
@tool
def code_of_conduct(query:str,whos:Literal["coc-student","coc-employee"]):
    """Get the code of conduct of student and employee"""
    docs=retriever(query=query,namespace=whos)
    return docs


In [51]:
@tool 
def rules_regulations(query:str):
    """Get the rules and regulations of the institute"""
    docs=retriever(query=query,namespace="rules-regulations")
    return docs

In [52]:
@tool
def placements():
    """Tool to get the information regarding placements"""
    result = tavily_extract.invoke(
        {"urls": ["https://ies.ipsacademy.org/training/"]}
    )
    return result["results"][0]

In [60]:
from langgraph.prebuilt import ToolNode

tools = [
    get_attendence,
    get_syllabus, 
    scrape_url,
    academic_calander,
    upcoming_events,
    academic_programs,
    admission_procedure,
    institute_brochure,
    code_of_conduct,
    rules_regulations,
    placements
]
toolnode = ToolNode(tools) 

In [54]:
from langchain_core.messages import SystemMessage


def chat(state: GlobalState):
    print('CHat')
    res = llm.bind_tools(tools).invoke(
        [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
    )
    # print(res)
    return {"messages": [res]}

In [55]:
checkpoint = MemorySaver()

In [ ]:
from langgraph.prebuilt import tools_condition
from langchain_core.messages import HumanMessage

graph = StateGraph(GlobalState)
graph.add_node("chat", chat)
graph.add_edge(START, "chat")
graph.add_node("tools", toolnode)
graph.add_conditional_edges("chat", tools_condition)
graph.add_edge("tools", "chat")
graph.add_edge("chat", END)
wf = graph.compile(checkpointer=checkpoint)
# wf = graph.compile()


In [58]:
while True:
    a = input("- ")
    if a == "`":
        break
    print("User - ",a)
    for i,j in wf.stream({"messages": [HumanMessage(content=a)]} ,config={'configurable':{'thread_id':0.11}},stream_mode="messages"):
        print(i.content, end='',flush=True)

    print("\n-------------------")

User -  library rules ?
CHat
["13\n3.7 Librarian\nIt is needless to overemphasize the fact that effective and efficient management of\nLibrary (including the key role played by the Librarian) is one of the indicative and\nparameters in the functional assessment of an educational institution.\nThe duties of the Librarian being multi-fold and vital fall under the following\nheads.\n Processing section.\n Circulation section.\n Reference section\n Journal section.\n Automation process.\n Reprographic section.\n Binding section.\n Budgeting.\n News papers & magazines section.\n Issuance of library cards.\n Up-keep & maintenance.\n Other related activities.\nThe Librarian shall take care of the following:\n Books, Magazines, Journals and such other literature donated shall be\nsuitably stamped, numbered, grouped and preserved properly.\n Books given to the staff at a time are limited to 8 books only. Books given at\na time for students is limited to 2 only for a period of two 

In [22]:
# result = tavily_extract.invoke(
#     {"urls": ["https://ies.ipsacademy.org/academics/academic-programs/b-e"]}
# )
# result

In [23]:
# result = app.scrape("https://ies.ipsacademy.org/wp-content/uploads/2016/12/Schedule-ODD-Semester-2026-27.pdf", formats=["html"])
# result
# scrape_id = result.metadata.scrape_id

# 2. Interact — search for a product and get its price
# res=app.interact(scrape_id, prompt="login using computer code - 70004 , pass- 11-01-2006 , and return mst marksheet")
# print(res)
#
# doc = app.scrape("https://cms2.ipsacademy.net/Student")
# response = app.interact(scrape_id, prompt="click on top right , select previus session then go to view report click regular result , submit , return the cgpa")
# print(response.output)
# app.stop_interaction(scrape_id)

In [24]:
# doc = app.scrape("https://ies.ipsacademy.org/wp-content/uploads/2016/12/CSERL-V-Sem-Scheme-and-Syllabus.pdf")

# print(res.output)